In [69]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

CSV_PATH = "raw_loan_dataset.csv"
df = pd.read_csv(CSV_PATH)
print(df.head())

# print(df.isnull().sum())

    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  


In [70]:
df["Income"] = df["Income"].replace(r"[$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[$,]", "", regex=True).astype(float)

print(df.head())

yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
print(df["Approved"].unique())
print(df["Approved"].value_counts(dropna=False))
print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0             Y   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  
['No' 'Yes']
Approved
Yes    68
No     35
Name: count, dtype: int64
     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0           

In [71]:

df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

print(df.head(5))
before = df.shape[0]
df = df.drop_duplicates()
print(f"{before}  -----  {df.shape[0]}")

     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0        638.0              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  
103  -----  100


In [17]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

# print(f"Income: Lower = {low_income:.2f}, Upper = {high_income:.2f}")
# print(f"CreditScore: Lower = {low_credit:.2f}, Upper = {high_credit:.2f}")
# print(f"LoanAmount: Lower = {low_loan:.2f}, Upper = {high_loan:.2f}")
# print(f"EmploymentYears: Lower = {low_emp:.2f}, Upper = {high_emp:.2f}")


Income: Lower = -23827.88, Upper = 167619.12
CreditScore: Lower = 344.25, Upper = 962.25
LoanAmount: Lower = -16687.50, Upper = 66212.50
EmploymentYears: Lower = -11.28, Upper = 35.53


In [72]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)
df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)

print(df.head(5))



     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0              1   
1   96482.0        524.0             15.0     11200.0              1   
2   28478.0        638.0              5.4     12100.0              0   
3   25851.0        561.0             17.6      7000.0              1   
4   38396.0        527.0              9.8     10700.0              0   

   PreviousDefaults  Approved  
0                 0         0  
1                 0         1  
2                 0         1  
3                 0         1  
4                 0         1  


In [33]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")


Class balance OK for baseline Accuracy (both classes well represented).


In [73]:
df["LoanToIncomeRatio"] = df["LoanAmount"] / df["Income"]

print(df.head(5))


     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0              1   
1   96482.0        524.0             15.0     11200.0              1   
2   28478.0        638.0              5.4     12100.0              0   
3   25851.0        561.0             17.6      7000.0              1   
4   38396.0        527.0              9.8     10700.0              0   

   PreviousDefaults  Approved  LoanToIncomeRatio  
0                 0         0           0.237111  
1                 0         1           0.116084  
2                 0         1           0.424889  
3                 0         1           0.270783  
4                 0         1           0.278675  


In [76]:
num_cols = ["Income", "CreditScore", "EmploymentYears", "LoanAmount"]

scaler = RobustScaler()

df[num_cols] = scaler.fit_transform(df[num_cols])


In [78]:
OUT_PATH = "clean_loan_dataset.csv"
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")


Saved cleaned dataset to clean_loan_dataset.csv
